# Step 4.4: the CFO ask. How much money can be stopped, reclaimed or avoided?

**The CFO's question** (`01` p5): where is the money going on expiry, and how much of it is recoverable?

**How we answer.** We take every claim lodged in the last 12 months (1 Oct 2025 to 30 Sep 2026, not rejected) and put it in **one of three groups, each claim once**, so the groups add up to the total:

| Group | What it is | What can be done |
|---|---|---|
| **G1 Breaks a written rule** | late (R1), more than supplied (R3), or a probable duplicate (R4) | stop it if pending; reclaim it if already credited; block it at intake in future |
| **G2 Linked to a scheme batch** | no rule broken, but the batch was invoiced on a quarter scheme | avoid it by changing scheme timing or size. We do not know by how much, so we show a **sensitivity**, not a forecast |
| **G3 The rest** | no rule broken, no scheme link | genuine expiry; not recoverable from the claim side |

**What we will not do.** We do not promise a recovery. A rule breach is a finding, not proof: the data has no field for an approved exception. Each amount below says what the client must confirm before it can be claimed.
Everything is at the claim value the client recorded (quantity times PTS). Nothing is profit.

Run top to bottom. Needs `step4_rules.ipynb` outputs. Nothing here changes a data file.

In [1]:
import os, sys
import numpy as np, pandas as pd
sys.path.insert(0, os.path.join('..', 'pilot'))
from agent.data import load
R = os.path.join('..', 'data-clean', 'audit', 'rules'); OUT = os.path.join('..', 'data-clean', 'audit', 'recovery'); os.makedirs(OUT, exist_ok=True)
ec = load('expiry_claims'); ps = load('primary_sales')
W0, W1 = pd.Timestamp('2025-10-01'), pd.Timestamp('2026-09-30')
cr = lambda x: f'{x/1e7:.2f} crore'
c = ec[ec.status != 'Rejected']; cl = c[(c.claim_date >= W0) & (c.claim_date <= W1)].copy()
cl['amount'] = np.where(cl.status == 'Credited', cl.credit_value_inr, cl.claim_value_derived_inr)   # paid, or still owed
print('claims in window:', len(cl), '| credited', (cl.status == 'Credited').sum(), '| pending', (cl.status == 'Pending').sum(), '| amount', cr(cl.amount.sum()))

claims in window: 2294 | credited 1934 | pending 360 | amount 5.44 crore


## Step 1. Which claims break a rule, and how much of each is at issue?
For each claim we take the amount at issue: the **whole claim** for a late claim (R1) or a probable duplicate (R4), and only the **excess packs** for an over-supply claim (R3).
**Firm** = R1 where the date is clear, R3, R4 probable duplicates. **Date-dependent** = R1 claims where the answer depends on which date is right (claim number month or claim date).

In [2]:
r1 = pd.read_csv(os.path.join(R, 'r1_breaches.csv')); r3 = pd.read_csv(os.path.join(R, 'r3_breaches.csv')); r4 = pd.read_csv(os.path.join(R, 'r4_repeat_claims.csv'))
r4 = r4[r4.r4_probable_dup]
whole_firm = set(r1[r1.r1_confidence == 'clear'].claim_no) | set(r4.claim_no)
whole_dep = set(r1[r1.r1_confidence != 'clear'].claim_no) - whole_firm
ex3 = r3.set_index('claim_no').r3_inr
cl['firm_at_issue'] = np.where(cl.claim_no.isin(whole_firm), cl.amount, np.where(cl.claim_no.isin(ex3.index), cl.claim_no.map(ex3).clip(upper=cl.amount), 0.0))
cl['dep_at_issue'] = np.where(cl.claim_no.isin(whole_dep) & (cl.firm_at_issue == 0), cl.amount, 0.0)
cl['g'] = np.where((cl.firm_at_issue > 0) | (cl.dep_at_issue > 0), 'G1', None)
print('claims firm:', (cl.firm_at_issue > 0).sum(), '| date-dependent only:', (cl.dep_at_issue > 0).sum())
print('at issue firm:', cr(cl.firm_at_issue.sum()), '| date-dependent:', cr(cl.dep_at_issue.sum()))
print('Check: each claim counted once; R3 counts only the excess packs.')
assert cl.claim_no.is_unique

claims firm: 261 | date-dependent only: 75
at issue firm: 0.32 crore | date-dependent: 0.16 crore
Check: each claim counted once; R3 counts only the excess packs.


## Step 2. Put every claim in one group and add them up

In [3]:
sup = ps[ps.batch_no.notna()].copy(); sup['sch'] = sup.scheme_code.fillna('NA') != 'NO_SCHEME'
sb = sup.groupby(['stockist_code', 'batch_no']).sch.max().reset_index()
cl = cl.merge(sb, on=['stockist_code', 'batch_no'], how='left')
assert cl.sch.notna().all()                  # every in-window claim traces to an invoice (the untraced ones are older, Jul 2024 to Mar 2025)
cl['group'] = np.where(cl.g == 'G1', 'G1 breaks a written rule', np.where(cl.sch == True, 'G2 linked to a scheme batch', 'G3 the rest'))
t = cl.groupby('group').agg(claims=('claim_no', 'size'), amount=('amount', 'sum'),
        credited=('amount', lambda s: s[cl.loc[s.index, 'status'] == 'Credited'].sum()),
        pending=('amount', lambda s: s[cl.loc[s.index, 'status'] == 'Pending'].sum()))
t.loc['Total'] = t.sum()
show = t.copy()
for k in ['amount', 'credited', 'pending']: show[k] = (show[k] / 1e7).round(2)
print(show); show.to_csv(os.path.join(OUT, 'groups.csv'))
assert abs(t.loc['Total', 'amount'] - cl.amount.sum()) < 1 and t.loc['Total', 'claims'] == len(cl)
print('Check: the three groups add up to the total; every claim is in exactly one group.')

                             claims  amount  credited  pending
group                                                         
G1 breaks a written rule        336    0.90      0.69     0.21
G2 linked to a scheme batch    1416    4.13      3.47     0.67
G3 the rest                     542    0.40      0.32     0.08
Total                          2294    5.44      4.48     0.96
Check: the three groups add up to the total; every claim is in exactly one group.


## Step 3. What the CFO can do with each group
- **Stop now.** Pending claims in G1 (firm). No credit note exists, so nothing has to be taken back.
- **Reclaim.** Credited claims in G1 (firm). Needs the client to agree to offset the amount against future credit notes. Some may be approved exceptions.
- **Prevent.** All G1 (firm) is the yearly flow a check at claim intake would have caught. This is what the pilot is built to do.
- **Avoid.** G2. We do not know how far a scheme change would reduce claims, so we show what each 10% would be worth.

In [4]:
g1 = cl[cl.g == 'G1']; firm_pending = g1[(g1.status == 'Pending')].firm_at_issue.sum(); firm_credited = g1[(g1.status == 'Credited')].firm_at_issue.sum()
dep = g1.dep_at_issue.sum(); g2 = cl[cl.group.str.startswith('G2')].amount.sum(); g3 = cl[cl.group.str.startswith('G3')].amount.sum()
A = pd.DataFrame([
 ('Stop now', 'pending claims that break a rule (firm)', firm_pending, firm_pending, firm_pending + g1[g1.status == 'Pending'].dep_at_issue.sum(), 'Client confirms no approved exception'),
 ('Reclaim', 'credited claims that break a rule (firm)', firm_credited, 0, firm_credited + g1[g1.status == 'Credited'].dep_at_issue.sum(), 'Client agrees to offset; checks for approved exceptions; settles which date is true'),
 ('Prevent every year', 'all firm breaches, caught at intake', firm_pending + firm_credited, firm_pending + firm_credited, firm_pending + firm_credited + dep, 'Pilot checks R1, R3, R4 when the claim arrives; some may be allowed by an exception we cannot see'),
 ('Avoid, per 10% fewer claims', 'claims linked to scheme batches (G2)', g2 * 0.10, g2 * 0.05, g2 * 0.20, 'A sensitivity, not a forecast. Needs a scheme change and the take-back terms (B7)'),
], columns=['lever', 'what', 'amount', 'low', 'high', 'client must confirm'])
A['crore'] = (A.amount / 1e7).round(2); A['range'] = (A.low / 1e7).round(2).astype(str) + ' to ' + (A.high / 1e7).round(2).astype(str)
pd.set_option('display.width', 250, 'display.max_colwidth', 80)
print(A[['lever', 'what', 'crore', 'range', 'client must confirm']].to_string(index=False)); A.to_csv(os.path.join(OUT, 'levers.csv'), index=False)
print(f'\nG2 total {cr(g2)}; G3 total {cr(g3)} (not recoverable from the claim side)')
tot = cl.amount.sum(); g1_amt = g1.amount.sum()
print(f'Of {cr(tot)} lodged in the window: G1 claims {cr(g1_amt)} ({g1_amt/tot*100:.0f}%), of which at issue {cr(firm_pending + firm_credited + dep)} (firm {cr(firm_pending + firm_credited)}); G2 {g2/tot*100:.0f}%; G3 {g3/tot*100:.0f}%')
print('At issue is smaller than the claim amount because for over-supply claims (R3) only the excess packs are wrong.')
assert firm_pending >= 0 and firm_credited >= 0
assert abs((g1_amt + g2 + g3) - tot) < 1 and (firm_pending + firm_credited + dep) <= g1_amt + 1
print('Check: G1 claims + G2 + G3 = all claims in the window; the amount at issue never exceeds the G1 claims.')

                      lever                                     what  crore        range                                                                               client must confirm
                   Stop now  pending claims that break a rule (firm)   0.06 0.06 to 0.14                                                             Client confirms no approved exception
                    Reclaim credited claims that break a rule (firm)   0.25  0.0 to 0.34               Client agrees to offset; checks for approved exceptions; settles which date is true
         Prevent every year      all firm breaches, caught at intake   0.32 0.32 to 0.48 Pilot checks R1, R3, R4 when the claim arrives; some may be allowed by an exception we cannot see
Avoid, per 10% fewer claims     claims linked to scheme batches (G2)   0.41 0.21 to 0.83                 A sensitivity, not a forecast. Needs a scheme change and the take-back terms (B7)

G2 total 4.13 crore; G3 total 0.40 crore (not recoverable from t

## Step 4. Questions the client must answer before any of this can be claimed
1. **Approved exceptions.** Is there a register of Area Manager or CFO approvals for late, over-supplied or no-invoice claims? Without it, G1 is a finding, not a recovery.
2. **Which date is true?** For 81 claims the claim number month and `claim_date` disagree (R12). One answer moves up to the date-dependent amount above.
3. **Offset.** May the company net a reclaimed amount against future credit notes to the stockist?
4. **Scheme take-back.** What do stockists get back on expiry under a scheme (B7)? Is any of the claim loss charged to the sales P&L?
5. **Breakage 7-day window.** How is the 7 days counted (R1)? About 77% of breakage and short claims fall outside 7 days of invoice, which looks too high.

In [5]:
print('ALL RECOVERY CHECKS PASSED; wrote', os.path.join(OUT, 'groups.csv'), 'and', os.path.join(OUT, 'levers.csv'))

ALL RECOVERY CHECKS PASSED; wrote ..\data-clean\audit\recovery\groups.csv and ..\data-clean\audit\recovery\levers.csv
